In [1]:
import pandas as pd
import re
import emoji



In [2]:
df = pd.read_csv('../datasets/train.csv')
df.head(10)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
5,00025465d4725e87,"""\n\nCongratulations from me as well, use the ...",0,0,0,0,0,0
6,0002bcb3da6cb337,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0
7,00031b1e95af7921,Your vandalism to the Matt Shirvington article...,0,0,0,0,0,0
8,00037261f536c51d,Sorry if the word 'nonsense' was offensive to ...,0,0,0,0,0,0
9,00040093b2687caa,alignment on this subject and which are contra...,0,0,0,0,0,0


In [3]:
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print("Label counts:")
print(df[label_cols].sum())
print(f"\nFully clean rows: {(df[label_cols].sum(axis=1) == 0).sum()}")

Label counts:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Fully clean rows: 143346


In [4]:
df['target'] = (
    df['severe_toxic']   * 0.40 +   # cancer wishes, graphic threats
    df['threat']         * 0.40 +   # death threats, violence
    df['identity_hate']  * 0.30 +   # targeted hate
    df['toxic']          * 0.25 +   # general toxicity
    df['insult']         * 0.20 +   # insults
    df['obscene']        * 0.10     # obscene but not necessarily harmful
).clip(0, 1)

print(f"Target range: {df['target'].min():.4f} – {df['target'].max():.4f}")
print(f"\nTarget distribution:")
print(pd.cut(df['target'], bins=[0, 0.20, 0.45, 0.75, 1.0],
             labels=['Safe', 'Ambiguous', 'Implicit/Covert Hate', 'Explicit Toxicity']).value_counts())

Target range: 0.0000 – 1.0000

Target distribution:
target
Ambiguous               8899
Implicit/Covert Hate    4481
Explicit Toxicity       2227
Safe                     618
Name: count, dtype: int64


In [5]:
def cleanText(text):
    if not isinstance(text, str):
        return str(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Collapse spacer obfuscation: "b i t c h" → "bitch"
    text = re.sub(r'\b(\w)\s(?=(\w\s){0,5}\w\b)', r'\1', text)

    cleaned = []
    for char in text:
        # Preserve alphanumeric, whitespace, emoji, and !?
        if char.isalnum() or char.isspace() or emoji.is_emoji(char) or char in '!?':
            cleaned.append(char)

    return ' '.join(''.join(cleaned).split())

print("Cleaning text...")
df['cleaned_text'] = df['comment_text'].apply(cleanText)
print(f"Done. Sample: {df['cleaned_text'].iloc[0][:100]}")

Cleaning text...
Done. Sample: Explanation Why the edits made under my username Hardcore Metallica Fan were reverted? They werent v


In [ ]:
check_phrases = [
    'i hope you die of cancer',
    'kill yourself',
    'i hope you get cancer and die',
    'drop dead',
]
print("Spot-checking threat phrases in dataset:")
for phrase in check_phrases:
    matches = df[df['cleaned_text'].str.contains(phrase, case=False, na=False)]
    if len(matches) > 0:
        print(f"\n'{phrase}' → {len(matches)} matches")
        print(matches[['cleaned_text', 'threat', 'severe_toxic', 'target']].head(3).to_string())
    else:
        print(f"\n'{phrase}' → not found in dataset")

Spot-checking threat phrases in new dataset:

'i hope you die of cancer' → 2 matches
                                                                                                              cleaned_text  threat  severe_toxic  target
13354  You fucking chink i hope you die of cancer like your grandmother did fucking jack off get the fuck off the internet       1             0    0.95
62619                                                                                             I hope you die of cancer       1             0    0.65

'kill yourself' → 46 matches
                                                                                                                                                  cleaned_text  threat  severe_toxic  target
2373  Go kill yourself You should be ashamed of yourself Twoofers like you are scumbags that deserve to die You antiscientific assholes are destroying America       1             0    0.95
2915                                            

In [7]:
df_toxic    = df[df['target'] > 0.001].copy()
df_safe     = df[df['target'] <= 0.001].sample(n=15000, random_state=42)
df_balanced = pd.concat([df_toxic, df_safe]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced dataset distribution:")
print(pd.cut(df_balanced['target'],
             bins=[0, 0.001, 0.20, 0.45, 0.75, 1.0],
             labels=['Exact Zero', 'Safe', 'Ambiguous',
                     'Implicit/Covert Hate', 'Explicit Toxicity']).value_counts())
print(f"Total: {len(df_balanced)} rows")

Balanced dataset distribution:
target
Ambiguous               8899
Implicit/Covert Hate    4481
Explicit Toxicity       2227
Safe                     618
Exact Zero                 0
Name: count, dtype: int64
Total: 31225 rows


In [8]:
import os
os.makedirs('../data/processed', exist_ok=True)

output_cols = ['comment_text', 'cleaned_text', 'target'] + label_cols
df_balanced[output_cols].rename(columns={'comment_text': 'text'}).to_csv(
    '../data/processed/train_cleaned.csv', index=False
)
print("Saved to ../data/processed/train_cleaned.csv")


df_check = pd.read_csv('../data/processed/train_cleaned.csv')
print(f"\nFinal file: {len(df_check)} rows, columns: {df_check.columns.tolist()}")
print(f"Target range: {df_check['target'].min():.4f} – {df_check['target'].max():.4f}")


Saved to ../data/processed/train_cleaned.csv

Final file: 31225 rows, columns: ['text', 'cleaned_text', 'target', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
Target range: 0.0000 – 1.0000


In [10]:
def score_to_category(score):
    if score <= 0.20:   return "Safe"
    elif score <= 0.45: return "Ambiguous"
    elif score <= 0.75: return "Implicit/Covert Hate"
    else:               return "Explicit Toxicity"

df = pd.read_csv('../data/processed/train_cleaned.csv')
print(f"Loaded {len(df)} rows | target range: {df['target'].min():.4f}–{df['target'].max():.4f}")


df['category'] = df['target'].apply(score_to_category)
print(df['category'].value_counts())
df.head(5)

Loaded 31225 rows | target range: 0.0000–1.0000
category
Safe                    15618
Ambiguous                8899
Implicit/Covert Hate     4481
Explicit Toxicity        2227
Name: count, dtype: int64


,text,cleaned_text,target,toxic,severe_toxic,obscene,threat,insult,identity_hate,category
0,"""==dont leave==\nWikipedia needs people like y...",dont leave Wikipedia needs people like you Rem...,0.00,0,0,0,0,0,0,Safe
1,First problem ParacusForward like most wiki ed...,First problem ParacusForward like most wiki ed...,0.25,1,0,0,0,0,0,Ambiguous
2,"Grow up, cyber yuppie.",Grow up cyber yuppie,0.25,1,0,0,0,0,0,Ambiguous
3,"""\nThis is ridiculous. I can't respond BECAUS...",This is ridiculous I cant respond BECAUSE YOU ...,0.00,0,0,0,0,0,0,Safe
4,Here is proof of notability \nCalton has aimed...,Here is proof of notability Calton has aimed h...,0.25,1,0,0,0,0,0,Ambiguous
